# EfficientNet-B0 (initial attempt): train on your own computer

The local version of `train_colab_effb0.ipynb`, for the **multi-task model only** (one backbone, crop head +
disease head). The independent crop and disease models are not trained here. It runs the same scripts
(`train.py` and `evaluate.py`) with the same config (`configs/baseline.yaml`), and saves to the same run name, so
results from here and from Colab can be compared and mixed.

| Run | Output folder |
|---|---|
| Multi-task (one model, crop head + disease head) | `outputs/multitask_effb0` |
| Evaluation | `outputs/multitask_effb0/eval` |

## Before the first run

1. **GPU:** an NVIDIA GPU with at least 4 GB of memory, and PyTorch with CUDA
   (`torch.cuda.is_available()` must be `True`). Training at batch 64 with mixed precision uses about 3.2 GB,
   so it fits on a 6 GB laptop GPU without splitting the batch. That matters for EfficientNet: it uses
   BatchNorm, so a smaller batch would change the results.
2. **Packages:** `pip install timm pyyaml pandas matplotlib scikit-learn tqdm`
3. **Dataset:** the 3 CSVs and the unpacked images go in the repo's `dataset/` folder:
   `dataset/DPD_Small_train.csv`, `dataset/DPD_Small_test_seen.csv`, `dataset/DPD_Small_test_unseen.csv`
   and `dataset/DPD_Small_dataset/*.jpg`.
4. The first run downloads the pretrained ImageNet weights (~21 MB) from Hugging Face. A warning about a missing
   `HF_TOKEN` is safe to ignore.

Close other programs that use the GPU, keep a laptop plugged in, and turn off sleep while training.

## If a finished run is already in `outputs/multitask_effb0`

Training is skipped and the notebook goes straight to evaluation and results. Set `FORCE_RETRAIN = True` to
train again. That **overwrites** the existing run, so copy the folder somewhere first if you want to keep it.

## If training stops

Run all cells again. A run that was cut off restarts from epoch 1, because `train.py` can't resume yet.
Pressing **Stop** in Jupyter also stops the training process.

## 1. Settings

In [ ]:
# ===== Edit these =====
FORCE_RETRAIN = False   # True = train again even if a finished run is already in outputs/ (overwrites it)
SHARE_DIR     = None    # optional: a synced team folder, e.g. "G:/My Drive/AML_Group_Project/aml_runs".
                        # None = make a zip to upload by hand instead (last cell).
# ======================

MODEL_DIR = "baseline/MultiTaskBaseline/InitialAttempt(Effb0)"
CONFIG    = "configs/baseline.yaml"
MODE      = "multitask"
EXP       = "multitask_effb0"      # same run name as the Colab notebook

## 2. Check the repo, GPU and dataset

The scripts use paths relative to the repo root, so this cell moves there first. The GPU and library versions are
printed so the report can say where the model was trained. The GPU is checked from a separate process, so this
notebook never holds GPU memory that training needs.

In [ ]:
import os
import platform
import subprocess
import sys

# Jupyter starts in this notebook's folder. Walk up to the repo root.
here = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(here, MODEL_DIR)):
    parent = os.path.dirname(here)
    if parent == here:
        raise FileNotFoundError(f"Can't find {MODEL_DIR}. Open this notebook from inside the project repo.")
    here = parent
os.chdir(here)
print("Repo root:", here)

r = subprocess.run(["git", "log", "-1", "--format=%h %s (%cr)"], capture_output=True, text=True)
print("Code version:", r.stdout.strip() or "unknown (git not found)")

PROBE = r"""
import torch
ok = torch.cuda.is_available()
free, total = torch.cuda.mem_get_info() if ok else (0, 0)
print(ok, torch.cuda.get_device_name(0) if ok else "-", free >> 20, total >> 20, torch.__version__, torch.version.cuda, sep="|")
"""
probe = subprocess.run([sys.executable, "-c", PROBE], capture_output=True, text=True)
assert probe.returncode == 0, f"Can't import torch in {sys.executable}:\n{probe.stderr}"
ok, GPU_NAME, free_mb, total_mb, torch_ver, cuda_ver = probe.stdout.strip().splitlines()[-1].split("|")
print(f"Python {platform.python_version()} ({sys.executable})")
print(f"torch {torch_ver} | CUDA {cuda_ver} | {platform.system()}")
assert ok == "True", ("No CUDA GPU found. This kernel has a CPU-only PyTorch: pick a kernel with CUDA PyTorch "
                      "(Select Kernel, top right) or install it, then run this cell again.")
free_gb, total_gb = int(free_mb) / 1024, int(total_mb) / 1024
print(f"GPU: {GPU_NAME} ({total_gb:.1f} GB, {free_gb:.1f} GB free)")
if free_gb < 4:
    print("Warning: under 4 GB free. Close programs that use the GPU (browser, games, video, other notebooks).")

import timm
print("timm", timm.__version__)

import pandas as pd

CSVS = ["DPD_Small_train.csv", "DPD_Small_test_seen.csv", "DPD_Small_test_unseen.csv"]
IMG_DIR = "dataset/DPD_Small_dataset"
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
missing = [n for n in CSVS if not os.path.isfile(f"dataset/{n}")]
assert not missing, f"Missing from dataset/: {missing}"
assert os.path.isdir(IMG_DIR), f"{IMG_DIR} not found. Unpack DPD_Small.7z there."

# Every image named in the CSVs must exist (first column; a header row, if any, is skipped).
names = pd.concat([pd.read_csv(f"dataset/{n}", header=None, usecols=[0])[0] for n in CSVS]).astype(str)
names = names[names.str.lower().str.endswith(IMG_EXT)]
have = set(os.listdir(IMG_DIR))
missing = [n for n in names if os.path.basename(n) not in have]
assert not missing, f"{len(missing)} images in the CSVs are missing, e.g. {missing[:3]}"
print(f"Images: {len(have)} (all {len(names)} CSV entries found)")

## 3. Helpers

`train()` and `evaluate()` run the scripts as separate processes, with live output here. That also avoids a
Windows problem where data-loader workers crash when training code runs inside Jupyter itself.

In [ ]:
import codecs
import csv

import yaml

os.makedirs("outputs", exist_ok=True)
with open(CONFIG) as f:
    CFG = yaml.safe_load(f)
EPOCHS, PATIENCE = CFG["train"]["epochs"], CFG["train"].get("patience", 0)
print(f"{CFG['model']['backbone']} | {CFG['data']['img_size']} px | batch {CFG['train']['batch_size']} | "
      f"up to {EPOCHS} epochs (early stop after {PATIENCE} without improvement)\n")


def run_status(exp):
    # 'done', 'partial' or 'missing' for a training run saved in outputs/.
    hist = f"outputs/{exp}/history.csv"
    if not (os.path.isfile(hist) and os.path.isfile(f"outputs/{exp}/best.pt")):
        return "missing"
    with open(hist) as f:
        scores = [float(r["val_score"]) for r in csv.DictReader(f)]
    if not scores:
        return "partial"
    best_epoch = scores.index(max(scores)) + 1          # same rule as train.py
    early_stop = PATIENCE and len(scores) - best_epoch >= PATIENCE
    return "done" if len(scores) >= EPOCHS or early_stop else "partial"


def run(args):
    # Run a script with this notebook's Python and stream its output. Stop in Jupyter stops it too.
    print("$", " ".join(args), flush=True)
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "PYTHONIOENCODING": "utf-8",
           "TQDM_MININTERVAL": "30"}   # fewer progress-bar updates keep the saved notebook small
    p = subprocess.Popen([sys.executable, *args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env)
    decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
    try:
        while chunk := p.stdout.read1(4096):
            sys.stdout.write(decoder.decode(chunk))
            sys.stdout.flush()
    except KeyboardInterrupt:
        if os.name == "nt":   # also stop the data-loader worker processes
            subprocess.run(["taskkill", "/F", "/T", "/PID", str(p.pid)], capture_output=True)
        else:
            p.terminate()
        p.wait()
        raise
    if p.wait():
        raise RuntimeError("Command failed, see the error above.")


def train():
    status = run_status(EXP)
    if status == "done" and not FORCE_RETRAIN:
        print(f"{EXP}: already finished, skipping. Set FORCE_RETRAIN = True to train it again.")
        return
    if status == "partial":
        print(f"{EXP}: found an unfinished run. Training restarts from epoch 1.")
    run([f"{MODEL_DIR}/train.py", "--config", CONFIG, "--mode", MODE, "--experiment", EXP])


def evaluate():
    if run_status(EXP) != "done":
        print(f"Skipping evaluation: training not finished for {EXP}")
        return
    ckpt = f"outputs/{EXP}/best.pt"
    metrics = f"outputs/{EXP}/eval/metrics.json"
    if (os.path.isfile(metrics) and not FORCE_RETRAIN
            and os.path.getmtime(metrics) > os.path.getmtime(ckpt)):
        print(f"{EXP}: already evaluated, skipping.")
        return
    run([f"{MODEL_DIR}/evaluate.py", "--checkpoints", ckpt, "--out", f"outputs/{EXP}/eval",
         "--batch_size", "64"])   # smaller than the default 128 to fit the GPU; doesn't change the metrics


print(f"{EXP:28s} {run_status(EXP)}")

## 4. Train the multi-task model

In [ ]:
train()

## 5. Evaluate on test_seen and test_unseen

In [ ]:
evaluate()

## 6. Results

In [ ]:
import json

import matplotlib.pyplot as plt
from IPython.display import Image, display

path = f"outputs/{EXP}/eval/metrics.json"
if os.path.isfile(path):
    with open(path) as f:
        m = json.load(f)
    row = {
        "seen crop": m["seen"]["crop"]["top1"],
        "seen disease": m["seen"]["disease"]["top1"],
        "seen pair": m["seen"]["composition"]["top1"],
        "unseen crop": m["unseen"]["crop"]["top1"],
        "unseen disease": m["unseen"]["disease"]["top1"],
        "unseen pair": m["unseen"]["composition"]["top1"],
        "HM": m["harmonic_mean"]["top1"],
        "best HM (calibrated)": m["calibrated"]["best_hm"],
        "AUC": m["calibrated"]["auc"],
    }
    print(f"Top-1 accuracy ({EXP})")
    display(pd.DataFrame({EXP: row}).T.round(4))
else:
    print("No evaluation results yet.")

# Training curves
hist = f"outputs/{EXP}/history.csv"
if os.path.isfile(hist):
    h = pd.read_csv(hist)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(h["epoch"], h["val_score"], label=EXP)
    ax[1].plot(h["epoch"], h["val_loss"], label=EXP)
    ax[0].set(title="Validation score (mean top-1 over heads)", xlabel="epoch")
    ax[1].set(title="Validation loss", xlabel="epoch")
    for a in ax:
        a.legend()
    plt.show()

for png in ["calibration_curve.png", "top_confusions_seen.png"]:
    path = f"outputs/{EXP}/eval/{png}"
    if os.path.isfile(path):
        print(path)
        display(Image(path, width=600))

## 7. Package the results for the team Drive

Collects the result folder with `best.pt` (the Colab notebook needs it to see the run as finished). `last.pt` is
left out to save space. With `SHARE_DIR` set, the folder is copied there. Otherwise it is zipped into
`outputs/share_effb0_local.zip`: upload it, then unzip the folder into `AML_Group_Project/aml_runs`.

The small result files (no checkpoints) can also be committed to git as usual. `.gitignore` already keeps the
`.pt` files out.

In [ ]:
import shutil
import zipfile

files = []
for root, _, fs in os.walk(f"outputs/{EXP}"):
    files += [os.path.join(root, fn) for fn in fs if fn != "last.pt"]
assert files, "No results yet. Run the training cell first."

if SHARE_DIR:
    for p in files:
        dst = os.path.join(SHARE_DIR, os.path.relpath(p, "outputs"))
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(p, dst)
    print(f"Copied {len(files)} files to {SHARE_DIR}")
else:
    zip_path = "outputs/share_effb0_local.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for p in files:
            z.write(p, os.path.relpath(p, "outputs"))
    print(f"{len(files)} files -> {os.path.abspath(zip_path)} ({os.path.getsize(zip_path) / 2**20:.0f} MB)")
print("Now save this notebook (Ctrl+S) and upload it too.")